# Pipeline Orchestrator

Central entry point for the PBM split-accumulator reconciliation pipeline.
Designed to be called by a Databricks Workflow on a daily schedule or run
manually for reconciliation.

| Stage | Notebook | Dependency |Gate behavior |
|-------|----------|--------|---------------|
| 1 | NB_01 file validation | Hard gate | pipeline halts on critical failure |
| 2 | NB_02 eligibility recon | Independent | failure logged, others continue |
| 2 | NB_03 accumulator recon | Independent | failure logged, others continue |
| 2 | NB_04 copay program audit | Independent | failure logged, others continue |
| 2 | NB_05 benefit year rollover | Independent | failure logged, others continue |
| 3 | NB_06 trend analysis | Non-blocking | failure does not affect findings |
| 4 | NB_07 convergence query | Depends on NB_01 through NB_05 | - |
| 4 | NB_08 executive dashboard | Depends on NB_07 | - |

NB_01 runs first because if the inbound file is fundamentally broken (empty, or 
more than 5% NULL member IDs), every downstream notebook produces garbage findings. 
Better to halt the pipeline and alert the team than to generate a hundred false
positives from corrupt data.

Non-critical warnings from NB_01 (schema drift, stale file) are logged
but do not block. The core notebooks (NB_02 through NB_05) each produce
findings independently, so a failure in one does not block the others.
NB_07's convergence query handles partial results.

Databricks Workflow setup:
1. Create a new Workflow with this notebook as the entry point
2. Schedule: daily at 07:00 UTC (after TPA file SLA window)
3. Cluster: shared job cluster with 4-8 workers
4. Alerts: email on failure (pager for NB_01 CRITICAL gate failures)
5. Retry policy: 1 retry with 5-minute delay for transient failures

Yes, I could create a Job or an ETL pipeline. But, for this purpose, I dediced to go with a pipeline orchestrator in a notebook.

In [0]:
%run ./NB_00_pipeline_utilities

# NB00 - Pipeline Utilities
**Eligibility & Accumulator**

In production, a reconciliation pipeline has dozens of checks spread across multiple notebooks.
If one those check fails, say the TPA eligibility table got dropped by an upstream job, you don't
want the entire pipeline to crash and leave you with nothing. You still want the accumulator
reconciliation, the copay audit, and the rollover checks to run and produce findings.

You also need to know exactly what ran, what passed, and what eventually broke.

This notebook provides shared utilities that every downstream notebook imports via
`%run ./NB_00_pipeline_utilities`. It handles three things:

| Utility | What it does |
|---------|-------------|
| Structured logging | JSON audit entries to `pbm_sim.pipeline_audit_log`, queryable by run, notebook, section |
| Error handling decorator | Wraps each check in try/except so one failure doesn't kill the entire pipeline |
| Table validation helpers | Pre-flight checks before a notebook starts its logic (do the table exists? does it has rows? does it have right columns?) |

These functions are used in NB01 through NB07.

## Structured Logging

Every pipeline run gets a unique `run_id` based on the UTC timestamp when the run started.
It ties together all the log entries from a single execution across notebooks.
If the orchestrator kicks off NB01 through NB07 sequentially, they all share the same run_id.

Log entries go to two places:
1. **stdout** as `[PIPELINE_LOG] {...}` - captured by Databricks job logs, useful for real-time monitoring
2. **Delta table** `pbm_sim.pipeline_audit_log` - the permanent audit trail you can query later

Each entry records the notebook, section, outcome (SUCCESS/WARNING/ERROR/SKIPPED),
row count, timing, and a free-text detail field. The detail field is truncated at 2,000
characters to prevent a full stack trace from bloating the table.

## Error Handling Decorator

This is the core safety mechanism. In a reconciliation pipeline, each notebook runs
multiple independent checks. Take NB03 (accumulator reconciliation) - it checks for
deductible overcharges AND OOPM overcharges. If the deductible check throws an exception
(maybe a column was renamed upstream), you still want the OOPM check to run and produce
its findings. Without this decorator, an unhandled exception in the deductible check
would kill the notebook, and you'd lose the OOPM results too.

The decorator does three things on failure:
1. Catches the exception and logs it as ERROR with the full traceback
2. Returns an empty DataFrame with the expected schema so downstream unions don't break
3. Lets the notebook continue to the next section

On success, it logs the section as SUCCESS with the row count and wall-clock time.

**Usage in a downstream notebook:**
```python
@pipeline_section("NB_03", "deductible_overcharge", finding_schema=convergence_schema)
def run_deductible_check():
    # ... compute df ...
    return df

df_result = run_deductible_check()
# If the check fails, df_result is an empty DataFrame with the right columns
# so the union at the end of the notebook still works
```

## Table Validation Helpers (Pre-flight Checks)

Before a notebook starts its actual logic, it should verify that the tables it depends on
exist and have data. Without this, a missing table produces a cryptic *AnalysisException*
50 cells into the notebook, and you spend 10 minutes figuring out that the problem was
simply that NB00 didn't run.

These are the pre-flight checks:
- **The Table exists?** Catches the case where the upstream notebook didn't run or the table got dropped
- **Are the table empty?** The table exists but has zero rows - maybe the data generator ran but the filter
  produced nothing. This is different from "table doesn't exist" and needs a different response
- **Are the required columns present?** Schema drift - a column was renamed or removed upstream. Better to catch
  this up front with a clear message than to get a "Column 'xyz' not found" error mid-analysis

## Pipeline Metadata

Convenience function for embedding run metadata into output tables.
When NB07 unions all findings, each row can carry the run_id so you can
trace any finding back to the exact pipeline execution that produced it.

Pipeline utilities loaded. Run ID: RUN_20261019_121149


In [0]:
import time

NB = "ORCHESTRATOR"
pipeline_start = time.time()

print(f"Pipeline run: {get_run_id()}")
print(f"Starting PBM reconciliation pipeline...\n")

Pipeline run: RUN_20261019_121149
Starting PBM reconciliation pipeline...



## Stage 1 - Inbound File Validation (NB_01)

This is the gate. If the inbound file is fundamentally broken, everything
downstream produces garbage. We run NB_01 first and check the results
before proceeding.

In [0]:
nb01_start = time.time()
try:
    dbutils.notebook.run("NB_01_inboud_file_validation", timeout_seconds=600)
    nb01_duration = time.time() - nb01_start
    log_entry(NB, "NB_01_file_validation", "SUCCESS", duration_sec=nb01_duration)
    print(f"NB_01 completed in {nb01_duration:.1f}s")
except Exception as e:
    nb01_duration = time.time() - nb01_start
    log_entry(NB, "NB_01_file_validation", "ERROR",
              detail=str(e), duration_sec=nb01_duration)
    raise RuntimeError(f"NB_01 file validation failed - pipeline halted: {e}")

[PIPELINE_LOG] {"run_id": "RUN_20261019_121149", "timestamp": "2026-10-19T12:12:22.029295+00:00", "notebook": "ORCHESTRATOR", "section": "NB_01_file_validation", "status": "SUCCESS", "row_count": 0, "detail": "", "duration_sec": 31.3}
NB_01 completed in 31.3s


In [0]:
# Gate check: verify the file is usable before continuing.
# Two hard thresholds - either one halts the pipeline:
#   1. Zero records means the file is empty (missed delivery or extract failure)
#   2. More than 5% NULL member_ids means the TPA export is broken
df_raw_gate = spark.table("pbm_sim.tpa_accumulator_file_raw")
total_records = df_raw_gate.count()
null_member_count = df_raw_gate.filter(F.col("member_id").isNull()).count()

if total_records == 0:
    log_entry(NB, "gate_check", "ERROR", detail="Raw file has 0 records - pipeline halted")
    raise RuntimeError("GATE FAILURE: Raw file is empty. Pipeline halted.")

null_rate = null_member_count / total_records
if null_rate > 0.05:
    log_entry(NB, "gate_check", "ERROR",
              detail=f"NULL member_id rate {null_rate:.1%} exceeds 5% threshold")
    raise RuntimeError(
        f"GATE FAILURE: {null_rate:.1%} of records have NULL member_id "
        f"({null_member_count}/{total_records}). Pipeline halted."
    )

# Record count reconciliation against the control file.
# A mismatch is a warning, not a gate failure - the file may have
# legitimately changed size, but operations should investigate.
df_ctrl = spark.table("pbm_sim.file_control_log")
if df_ctrl.count() > 0:
    latest_ctrl = df_ctrl.orderBy(F.col("file_date").desc()).first()
    if latest_ctrl and "expected_record_count" in df_ctrl.columns:
        expected = latest_ctrl["expected_record_count"]
        if expected and abs(total_records - expected) > 0:
            log_entry(NB, "gate_check", "WARNING",
                      detail=f"Record count mismatch: file has {total_records}, "
                             f"control expects {expected}")
            print(f"WARNING: Record count mismatch ({total_records} vs {expected} expected)")

log_entry(NB, "gate_check", "SUCCESS",
          detail=f"File passed gate: {total_records} records, "
                 f"{null_rate:.1%} NULL rate")
print(f"Gate passed: {total_records} records, {null_rate:.1%} NULL rate\n")

[PIPELINE_LOG] {"run_id": "RUN_20261019_121149", "timestamp": "2026-10-19T12:12:25.660425+00:00", "notebook": "ORCHESTRATOR", "section": "gate_check", "status": "WARNING", "row_count": 0, "detail": "Record count mismatch: file has 45617, control expects 45602", "duration_sec": 0.0}
[PIPELINE_LOG] {"run_id": "RUN_20261019_121149", "timestamp": "2026-10-19T12:12:27.044130+00:00", "notebook": "ORCHESTRATOR", "section": "gate_check", "status": "SUCCESS", "row_count": 0, "detail": "File passed gate: 45617 records, 0.0% NULL rate", "duration_sec": 0.0}
Gate passed: 45617 records, 0.0% NULL rate



## Stage 2 - Core Reconciliation (NB_02 through NB_05)

These four notebooks run in sequence. Each one produces findings
independently. A failure in one notebook is logged but does not block
the others - the convergence query (NB_07) handles partial results
by working with whatever findings tables exist. 

In [0]:
core_notebooks = [
    ("NB_02_eligibility_reconciliation", "NB_02_eligibility"),
    ("NB_03_accumulator_reconciliation", "NB_03_accumulator"),
    ("NB_04_copay_accumulator_audit",    "NB_04_copay_program"),
    ("NB_05_benefit_year_rollover",      "NB_05_rollover"),
]

core_results = {}
for nb_name, log_label in core_notebooks:
    nb_start = time.time()
    try:
        dbutils.notebook.run(nb_name, timeout_seconds=900)
        duration = time.time() - nb_start
        log_entry(NB, log_label, "SUCCESS", duration_sec=duration)
        core_results[nb_name] = "SUCCESS"
        print(f"  {log_label}: completed in {duration:.1f}s")
    except Exception as e:
        duration = time.time() - nb_start
        log_entry(NB, log_label, "ERROR", detail=str(e), duration_sec=duration)
        core_results[nb_name] = f"ERROR: {e}"
        print(f"  {log_label}: FAILED ({e})")

# Report core stage results
succeeded = sum(1 for v in core_results.values() if v == "SUCCESS")
failed = len(core_results) - succeeded
print(f"\nCore reconciliation: {succeeded}/{len(core_results)} notebooks succeeded")
if failed > 0:
    print(f"WARNING: {failed} notebook(s) failed - convergence query will have partial data")

[PIPELINE_LOG] {"run_id": "RUN_20261019_121149", "timestamp": "2026-10-19T12:13:20.462664+00:00", "notebook": "ORCHESTRATOR", "section": "NB_02_eligibility", "status": "SUCCESS", "row_count": 0, "detail": "", "duration_sec": 51.49}
  NB_02_eligibility: completed in 51.5s
[PIPELINE_LOG] {"run_id": "RUN_20261019_121149", "timestamp": "2026-10-19T12:14:05.418755+00:00", "notebook": "ORCHESTRATOR", "section": "NB_03_accumulator", "status": "SUCCESS", "row_count": 0, "detail": "", "duration_sec": 41.23}
  NB_03_accumulator: completed in 41.2s
[PIPELINE_LOG] {"run_id": "RUN_20261019_121149", "timestamp": "2026-10-19T12:14:58.372386+00:00", "notebook": "ORCHESTRATOR", "section": "NB_04_copay_program", "status": "SUCCESS", "row_count": 0, "detail": "", "duration_sec": 51.39}
  NB_04_copay_program: completed in 51.4s
[PIPELINE_LOG] {"run_id": "RUN_20261019_121149", "timestamp": "2026-10-19T12:15:41.334013+00:00", "notebook": "ORCHESTRATOR", "section": "NB_05_rollover", "status": "SUCCESS", "row

## Stage 3 - Trend Analysis (NB_06)

Builds the daily reconciliation log and runs spike detection.
Non-blocking - a failure here does not affect the findings registry
or the convergence query. The trend analysis is for operational
monitoring, not for the audit findings themselves.

In [0]:
nb06_start = time.time()
try:
    dbutils.notebook.run("NB_06_trend_analysis_time_travel", timeout_seconds=600)
    nb06_duration = time.time() - nb06_start
    log_entry(NB, "NB_06_trend_analysis", "SUCCESS", duration_sec=nb06_duration)
    print(f"NB_06 completed in {nb06_duration:.1f}s")
except Exception as e:
    nb06_duration = time.time() - nb06_start
    log_entry(NB, "NB_06_trend_analysis", "ERROR", detail=str(e), duration_sec=nb06_duration)
    print(f"NB_06 failed (non-blocking): {e}")

[PIPELINE_LOG] {"run_id": "RUN_20261019_121149", "timestamp": "2026-10-19T12:16:34.476397+00:00", "notebook": "ORCHESTRATOR", "section": "NB_06_trend_analysis", "status": "SUCCESS", "row_count": 0, "detail": "", "duration_sec": 51.5}
NB_06 completed in 51.5s


## Stage 4 - Convergence and Dashboard (NB_07, NB_08)

NB_07 unions all findings into the canonical registry and runs the
member-level and plan-level rollups. NB_08 reads the registry and
renders the executive dashboard. If NB_07 fails, we raise because
the dashboard would be working from stale data.

In [0]:
nb07_start = time.time()
try:
    dbutils.notebook.run("NB_07_convergence_query", timeout_seconds=900)
    nb07_duration = time.time() - nb07_start
    log_entry(NB, "NB_07_convergence", "SUCCESS", duration_sec=nb07_duration)
    print(f"NB_07 completed in {nb07_duration:.1f}s")
except Exception as e:
    nb07_duration = time.time() - nb07_start
    log_entry(NB, "NB_07_convergence", "ERROR", detail=str(e), duration_sec=nb07_duration)
    print(f"NB_07 failed: {e}")
    raise RuntimeError(f"NB_07 convergence query failed - dashboard will be stale: {e}")

[PIPELINE_LOG] {"run_id": "RUN_20261019_121149", "timestamp": "2026-10-19T12:19:39.486241+00:00", "notebook": "ORCHESTRATOR", "section": "NB_07_convergence", "status": "SUCCESS", "row_count": 0, "detail": "", "duration_sec": 182.05}
NB_07 completed in 182.1s


In [0]:
nb08_start = time.time()
try:
    dbutils.notebook.run("NB_08_executive_dashboard", timeout_seconds=600)
    nb08_duration = time.time() - nb08_start
    log_entry(NB, "NB_08_dashboard", "SUCCESS", duration_sec=nb08_duration)
    print(f"NB_08 completed in {nb08_duration:.1f}s")
except Exception as e:
    nb08_duration = time.time() - nb08_start
    log_entry(NB, "NB_08_dashboard", "ERROR", detail=str(e), duration_sec=nb08_duration)
    print(f"NB_08 failed (non-blocking): {e}")

[PIPELINE_LOG] {"run_id": "RUN_20261019_121149", "timestamp": "2026-10-19T12:20:32.606770+00:00", "notebook": "ORCHESTRATOR", "section": "NB_08_dashboard", "status": "SUCCESS", "row_count": 0, "detail": "", "duration_sec": 51.37}
NB_08 completed in 51.4s


## Pipeline Summary

In [0]:
pipeline_duration = time.time() - pipeline_start

print("=" * 65)
print("  PIPELINE EXECUTION SUMMARY")
print("=" * 65)
print(f"  Run ID:       {get_run_id()}")
print(f"  Duration:     {pipeline_duration:.1f}s ({pipeline_duration/60:.1f} min)")
print()
print("  Stage Results:")
print(f"    NB_01 File Validation:     PASSED (gate check OK)")
for nb_name, log_label in core_notebooks:
    status = core_results.get(nb_name, "UNKNOWN")
    icon = "+" if status == "SUCCESS" else "X"
    print(f"    [{icon}] {log_label}")
print()

# Read the convergence results if available
try:
    df_final = spark.table("pbm_sim.findings_registry")
    total_findings = df_final.count()
    total_exposure = df_final.agg(F.sum("dollar_exposure")).collect()[0][0] or 0
    critical = df_final.filter(F.col("severity") == "CRITICAL").count()
    members = df_final.select("member_id").distinct().count()

    print(f"  Pipeline Output:")
    print(f"    Total findings:     {total_findings}")
    print(f"    Members affected:   {members}")
    print(f"    Dollar exposure:    ${total_exposure:,.2f}")
    print(f"    CRITICAL findings:  {critical}")
except Exception:
    print("  (Could not read final results)")

print("=" * 65)

  PIPELINE EXECUTION SUMMARY
  Run ID:       RUN_20261019_121149
  Duration:     523.7s (8.7 min)

  Stage Results:
    NB_01 File Validation:     PASSED (gate check OK)
    [+] NB_02_eligibility
    [+] NB_03_accumulator
    [+] NB_04_copay_program
    [+] NB_05_rollover

  Pipeline Output:
    Total findings:     7084
    Members affected:   5851
    Dollar exposure:    $3,688,214.20
    CRITICAL findings:  2556


In [0]:
log_entry(NB, "pipeline_complete", "SUCCESS",
          detail=f"Duration: {pipeline_duration:.1f}s, "
                 f"Core: {succeeded}/{len(core_results)} succeeded",
          duration_sec=pipeline_duration)
log_summary(NB)

[PIPELINE_LOG] {"run_id": "RUN_20261019_121149", "timestamp": "2026-10-19T12:20:37.082084+00:00", "notebook": "ORCHESTRATOR", "section": "pipeline_complete", "status": "SUCCESS", "row_count": 0, "detail": "Duration: 523.7s, Core: 4/4 succeeded", "duration_sec": 523.74}

  PIPELINE LOG SUMMARY: ORCHESTRATOR (run: RUN_20261019_121149)
  [+] NB_01_file_validation: SUCCESS [31.3s]
  [!] gate_check: WARNING
      Record count mismatch: file has 45617, control expects 45602
  [+] gate_check: SUCCESS
  [+] NB_02_eligibility: SUCCESS [51.49s]
  [+] NB_03_accumulator: SUCCESS [41.23s]
  [+] NB_04_copay_program: SUCCESS [51.39s]
  [+] NB_05_rollover: SUCCESS [41.35s]
  [+] NB_06_trend_analysis: SUCCESS [51.5s]
  [+] NB_07_convergence: SUCCESS [182.05s]
  [+] NB_08_dashboard: SUCCESS [51.37s]
  [+] pipeline_complete: SUCCESS [523.74s]

